In [21]:
import pandas as pd
import numpy as np
import os

data_path = "../data/raw"

datasets = {}

for file in os.listdir(data_path):

    if file.endswith(".csv"):

        name = file.replace(".csv", "")

        datasets[name] = pd.read_csv(
            os.path.join(data_path, file)
        )

In [22]:
missing_report = []

for name, df in datasets.items():

    for column in df.columns:

        missing_count = df[column].isnull().sum()

        if missing_count > 0:

            missing_report.append({
                "Dataset": name,
                "Column": column,
                "Missing Count": missing_count,
                "Missing Percentage": round(
                    (missing_count / len(df)) * 100, 2
                )
            })

missing_df = pd.DataFrame(missing_report)

missing_df.sort_values(
    by="Missing Count",
    ascending=False
)

,Dataset,Column,Missing Count,Missing Percentage
3,olist_order_reviews_dataset,review_comment_title,87656,88.34
4,olist_order_reviews_dataset,review_comment_message,58247,58.70
2,olist_orders_dataset,order_delivered_customer_date,2965,2.98
1,olist_orders_dataset,order_delivered_carrier_date,1783,1.79
5,olist_products_dataset,product_category_name,610,1.85
7,olist_products_dataset,product_description_lenght,610,1.85
6,olist_products_dataset,product_name_lenght,610,1.85
8,olist_products_dataset,product_photos_qty,610,1.85
0,olist_orders_dataset,order_approved_at,160,0.16
9,olist_products_dataset,product_weight_g,2,0.01


In [23]:
orders = datasets["olist_orders_dataset"]

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [24]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [25]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [26]:
geo = datasets["olist_geolocation_dataset"]

print("Original rows:", len(geo))

print("Duplicate rows:", geo.duplicated().sum())

print("Unique ZIP codes:",
      geo["geolocation_zip_code_prefix"].nunique())

Original rows: 1000163
Duplicate rows: 261831
Unique ZIP codes: 19015


In [27]:
orders = datasets["olist_orders_dataset"].copy()

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [28]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [29]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [30]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [31]:
orders.groupby("order_status")[
    date_columns
].apply(lambda x: x.isnull().sum())

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
order_status,,,,,
approved,0,0,2,2,0
canceled,0,141,550,619,0
created,0,5,5,5,0
delivered,0,14,2,8,0
invoiced,0,0,314,314,0
processing,0,0,301,301,0
shipped,0,0,0,1107,0
unavailable,0,0,609,609,0


Clean the Products dataset

In [32]:
products = datasets["olist_products_dataset"].copy()

products.isnull().sum()

product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [33]:
(products.isnull().mean() * 100).round(2)

product_id                    0.00
product_category_name         1.85
product_name_lenght           1.85
product_description_lenght    1.85
product_photos_qty            1.85
product_weight_g              0.01
product_length_cm             0.01
product_height_cm             0.01
product_width_cm              0.01
dtype: float64

In [34]:
products["product_category_name"].value_counts(
    dropna=False
).head(15)

product_category_name
cama_mesa_banho                3029
esporte_lazer                  2867
moveis_decoracao               2657
beleza_saude                   2444
utilidades_domesticas          2335
automotivo                     1900
informatica_acessorios         1639
brinquedos                     1411
relogios_presentes             1329
telefonia                      1134
bebes                           919
perfumaria                      868
papelaria                       849
fashion_bolsas_e_acessorios     849
cool_stuff                      789
Name: count, dtype: int64

Clean the Reviews dataset

In [35]:
reviews = datasets["olist_order_reviews_dataset"].copy()

reviews.isnull().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [36]:
reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

Investigate Geolocation duplicates

In [37]:
geo = datasets["olist_geolocation_dataset"].copy()

geo.duplicated().sum()

np.int64(261831)

In [38]:
geo_clean = geo.groupby(
    "geolocation_zip_code_prefix",
    as_index=False
).agg({
    "geolocation_lat": "mean",
    "geolocation_lng": "mean",
    "geolocation_city": "first",
    "geolocation_state": "first"
})

In [39]:
for name, df in datasets.items():

    print("\nDataset:", name)

    print("Duplicate rows:", df.duplicated().sum())


Dataset: olist_customers_dataset
Duplicate rows: 0

Dataset: olist_geolocation_dataset
Duplicate rows: 261831

Dataset: olist_orders_dataset
Duplicate rows: 0

Dataset: olist_order_items_dataset
Duplicate rows: 0

Dataset: olist_order_payments_dataset
Duplicate rows: 0

Dataset: olist_order_reviews_dataset
Duplicate rows: 0

Dataset: olist_products_dataset
Duplicate rows: 0

Dataset: olist_sellers_dataset
Duplicate rows: 0


In [40]:
processed_path = "../data/processed"

os.makedirs(processed_path, exist_ok=True)

cleaned_datasets = {
    "customers": datasets["olist_customers_dataset"].copy(),
    "orders": orders,
    "order_items": datasets["olist_order_items_dataset"].copy(),
    "payments": datasets["olist_order_payments_dataset"].copy(),
    "reviews": reviews,
    "products": products,
    "sellers": datasets["olist_sellers_dataset"].copy(),
    "geolocation": geo_clean
}

for name, df in cleaned_datasets.items():

    df.to_csv(
        f"{processed_path}/{name}.csv",
        index=False
    )

print("Processed datasets saved successfully!")

Processed datasets saved successfully!
